# OpenFOAM CFD Setup — Toroidal Propeller Simulation

This notebook:
1. Installs OpenFOAM v2212 on Google Colab (Ubuntu Linux only — see note below)
2. Generates propeller STL geometry
3. Creates OpenFOAM case directories
4. Runs a representative case (medium-gap toroidal, 4000 RPM)
5. Post-processes forces and compares with BEMT / experimental data

> **⚠️ WINDOWS USERS:** OpenFOAM does NOT run on Windows natively.
> Running cell 2 will detect this and print step-by-step instructions for
> using **Google Colab** (recommended) or **WSL2**.
> The BEMT notebook (`01_BEMT_Analysis.ipynb`) **does** work on Windows.

---

## Experimental setup being reproduced

The experiments in the Palileo & Sabanal (2024) thesis were conducted in a
**Westenberg Engineering WT 8600100-E Eiffel-type open-jet subsonic wind tunnel**
located at PAG-ASA Davao:

| Parameter | Value |
|---|---|
| Tunnel type | Eiffel-type (open-circuit, open-jet) |
| Jet outlet diameter | **600 mm** |
| Max flow speed | 80 m/s |
| Tested speeds | 3, 9, 15 m/s |
| Propeller diameter | 203.2 mm (8 in) |
| Blockage ratio | ~9% (propeller disk / jet cross-section) |

The `wind_tunnel_openjet` case type in this notebook replicates this geometry:
the domain is 600 × 600 mm in cross-section (matching the jet), with slip lateral
boundaries approximating the free-jet shear layer.  This is physically more
accurate than a free-field simulation because the 9% blockage is significant.

---
**Solver:** simpleFoam (steady-state RANS)  
**Turbulence:** k-ω SST  
**Rotation:** Multiple Reference Frame (MRF)

In [ ]:
import os, sys

REPO_URL = 'https://github.com/rp3gregorio/Propeller-CFD.git'
BRANCH   = 'claude/add-cfd-propeller-simulation-EEQqr'

# ── Detect environment ──────────────────────────────────────────────────
ON_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')

if ON_COLAB:
    REPO_DIR = '/content/Propeller-CFD'
    if not os.path.isdir(REPO_DIR):
        !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} fetch origin
        !git -C {REPO_DIR} checkout {BRANCH}
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    # Running locally — find the repo root by looking for bemt/ directory
    _here = os.path.abspath(os.getcwd())
    for _candidate in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_candidate, 'bemt')):
            REPO_DIR = _candidate
            break
    else:
        raise RuntimeError(
            "Could not find the repo root.\n"
            "Open Jupyter from inside the cloned Propeller-CFD folder."
        )
    print(f'Running locally. Repo root: {REPO_DIR}')

# Clear any cached Python imports
for _mod in [m for m in sys.modules if m.startswith(('bemt', 'geometry', 'postprocessing'))]:
    del sys.modules[_mod]

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import subprocess as _sp
print('Working directory:', os.getcwd())
print('Branch:', _sp.run('git branch --show-current', shell=True, capture_output=True, text=True, cwd=REPO_DIR).stdout.strip())
print('Latest commit:', _sp.run('git log --oneline -1', shell=True, capture_output=True, text=True, cwd=REPO_DIR).stdout.strip())

In [ ]:
import subprocess, os, sys, platform

# ── WINDOWS CHECK ─────────────────────────────────────────────────────────────
if platform.system() == 'Windows':
    raise RuntimeError(
        "\n"
        "╔══════════════════════════════════════════════════════════════════════╗\n"
        "║  OpenFOAM does NOT run on Windows natively.                         ║\n"
        "║                                                                      ║\n"
        "║  OPTION A — Google Colab (easiest, free):                            ║\n"
        "║    1. Go to https://colab.research.google.com                        ║\n"
        "║    2. File → Open notebook → GitHub tab                              ║\n"
        "║    3. Paste: rp3gregorio/Propeller-CFD                               ║\n"
        "║    4. Select branch: claude/add-cfd-propeller-simulation-EEQqr       ║\n"
        "║    5. Open: notebooks/02_OpenFOAM_Setup.ipynb                        ║\n"
        "║                                                                      ║\n"
        "║  The BEMT notebook (01_BEMT_Analysis.ipynb) works on Windows.       ║\n"
        "╚══════════════════════════════════════════════════════════════════════╝\n"
    )

# ── Already installed check ────────────────────────────────────────────────────
def _run(cmd, check=True, cwd=None):
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError(f'Command failed (rc={r.returncode}):\n{r.stdout[-2000:]}')
    return r

if _run('command -v blockMesh', check=False).returncode == 0 or \
   os.path.isfile('/usr/lib/openfoam/openfoam2212/bin/blockMesh'):
    print('OpenFOAM already installed — skipping.')
else:
    print('--- Installing OpenFOAM v2212 via official repo setup script ---')

    # Step 1: Download the official repo setup script to a temp file
    # (avoids the piped-curl approach which can fail if the server returns HTML)
    _run('curl -fsSL https://dl.openfoam.com/add-debian-repo.sh -o /tmp/add-of-repo.sh')

    # Step 2: Verify it looks like a shell script, not an error page
    with open('/tmp/add-of-repo.sh') as f:
        first_line = f.readline().strip()
    if not first_line.startswith('#'):
        raise RuntimeError(
            f'Downloaded script does not look like a shell script '
            f'(first line: {first_line!r}).\n'
            'Check your network connection / Colab internet access.'
        )

    # Step 3: Run the setup script (adds GPG key + apt source)
    print(_run('bash /tmp/add-of-repo.sh').stdout)

    # Step 4: Install OpenFOAM
    print('--- Installing openfoam2212-default (takes 5-8 min) ---')
    print(_run('apt-get install -y openfoam2212-default 2>&1 | tail -10').stdout)

    # Step 5: Verify
    r = _run('bash -c "source /usr/lib/openfoam/openfoam2212/etc/bashrc && foamVersion"',
             check=False)
    if r.returncode == 0:
        print(f'OpenFOAM version: {r.stdout.strip()}')
        print('Installation complete.')
    else:
        print('WARNING: foamVersion check failed — check output above.')
        print(r.stdout)

In [ ]:
# ── Step 3: Load OpenFOAM environment ─────────────────────────────────
import subprocess, os, sys, time

OF_BASHRC = '/usr/lib/openfoam/openfoam2212/etc/bashrc'

if not os.path.isfile(OF_BASHRC):
    raise RuntimeError(
        "OpenFOAM bashrc not found — install in cell 2 did not complete.\n"
        "→ Runtime → Disconnect and delete runtime → Runtime → Run all"
    )

# Source bashrc ONCE and capture the full resulting environment
_raw = subprocess.run(
    f'bash -c "source {OF_BASHRC} && env"',
    shell=True, capture_output=True, text=True
)
OF_ENV = {}
for _line in _raw.stdout.splitlines():
    _k, _, _v = _line.partition('=')
    if _k:
        OF_ENV[_k] = _v

_ver = OF_ENV.get('WM_PROJECT_VERSION', '')
_dir = OF_ENV.get('WM_PROJECT_DIR', '')
if not _ver:
    raise RuntimeError(
        "OpenFOAM environment did not load — WM_PROJECT_VERSION is unset.\n"
        "→ Runtime → Disconnect and delete runtime → Runtime → Run all"
    )
print(f'OpenFOAM {_ver} ready  ({_dir})')


def run_of(cmd, cwd=None):
    """Run a quick OpenFOAM command and return output (no streaming)."""
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        capture_output=True, text=True, env=OF_ENV
    )
    if result.returncode != 0:
        raise RuntimeError(
            f'OpenFOAM command failed: {cmd}\n'
            f'STDOUT: {result.stdout[-1500:]}\n'
            f'STDERR: {result.stderr[-1500:]}'
        )
    return result.stdout


def run_of_stream(cmd, cwd=None):
    """
    Run a long OpenFOAM command and stream output line-by-line in real time.
    Use this for blockMesh, snappyHexMesh, simpleFoam — anything that takes
    more than a few seconds, so you can see it is making progress.
    """
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env=OF_ENV, bufsize=1
    )
    lines = []
    for line in proc.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    proc.wait()
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(
            f'OpenFOAM command failed (rc={proc.returncode}): {cmd}\n'
            f'See output above for the FOAM error.'
        )
    print(f'\n✓ {cmd.split()[0]} finished in {elapsed/60:.1f} min')
    return ''.join(lines)

In [ ]:
# ── Step 4: Generate propeller STL geometry ─────────────────────────────
!pip install -q numpy
!python geometry/generate_stl.py

import os
stl_files = os.listdir('geometry/stl')
print('Generated STL files:', stl_files)

In [ ]:
# ── Step 5: Create a representative OpenFOAM case ──────────────────────
# Reproducing the Palileo & Sabanal (2024) wind tunnel experiment:
#   Westenberg WT 8600100-E open-jet tunnel, D_jet=600 mm, at V_inf=9 m/s
#   Medium-gap toroidal propeller (1.5 in gap) at 4000 RPM
#
# Change these parameters to simulate other experimental conditions:
#   CASE_TYPE options:
#     "wind_tunnel_openjet" — matches the actual Eiffel-type open-jet tunnel
#                             (600-mm jet, 9% blockage) ← RECOMMENDED for experiment comparison
#     "wind_tunnel"         — generic free-field domain (±10D, slip sides)
#     "static_thrust"       — quiescent air (V_inf=0, large open domain)
#
#   V_INF options tested experimentally: 3.0, 9.0, 15.0 m/s

CONFIG    = 'toroidal_medium_gap'   # best performer in experiments
CASE_TYPE = 'wind_tunnel_openjet'   # Eiffel-type open-jet tunnel (matches experiment)
RPM       = 4000                    # 1000 – 6000
V_INF     = 9.0                     # 3.0 / 9.0 / 15.0 m/s  (0.0 for static thrust)
N_PROCS   = 2                       # Colab has 2 CPUs

!python openfoam/generate_case.py \
    --config {CONFIG} \
    --case_type {CASE_TYPE} \
    --rpm {RPM} \
    --v_inf {V_INF} \
    --n_procs {N_PROCS}

CASE_DIR = f'runs/{CONFIG}_{RPM}rpm_{CASE_TYPE}_{int(V_INF)}ms'
print('Case directory:', CASE_DIR)
!ls {CASE_DIR}

In [ ]:
# ── Step 6: Run blockMesh ──────────────────────────────────────────────
# Expected time: ~10-30 seconds
print('Running blockMesh...')
run_of_stream('blockMesh', cwd=CASE_DIR)

In [ ]:
# ── Step 7: Run snappyHexMesh ─────────────────────────────────────────
# Expected time on Colab (2 CPUs): 10–20 minutes — this is normal.
# You will see live output below as each meshing phase runs:
#   Phase 1 — Castellation (cell splitting around the propeller)
#   Phase 2 — Snapping     (projecting mesh faces onto the STL surface)
#   Phase 3 — Layer adding (adding boundary-layer cells on the propeller wall)
print('Running snappyHexMesh — watch the live output below...')
print('(10–20 min on Colab is normal)\n')
run_of_stream('snappyHexMesh -overwrite', cwd=CASE_DIR)

In [ ]:
# ── Step 8: Check mesh quality ────────────────────────────────────────
out = run_of('checkMesh', cwd=CASE_DIR)
# Extract key mesh quality metrics
for line in out.split('\n'):
    if any(k in line for k in ['cells:', 'Max skewness', 'Max non-ortho', 'Overall']):
        print(line.strip())

In [ ]:
# ── Step 9: Run simpleFoam solver ─────────────────────────────────────
# Expected time on Colab: 20–60 minutes depending on mesh size and convergence.
# You will see residuals printed every iteration — watch for them to drop below 1e-4.
import shutil

# Copy initial conditions
orig = os.path.join(CASE_DIR, '0.orig')
zero = os.path.join(CASE_DIR, '0')
if os.path.isdir(orig) and not os.path.isdir(zero):
    shutil.copytree(orig, zero)

print('Running simpleFoam — residuals will stream below...')
print('(convergence typically takes 200–500 iterations)\n')
if N_PROCS > 1:
    run_of_stream('decomposePar', cwd=CASE_DIR)
    run_of_stream(f'mpirun -np {N_PROCS} simpleFoam -parallel 2>&1', cwd=CASE_DIR)
    run_of_stream('reconstructPar', cwd=CASE_DIR)
else:
    run_of_stream('simpleFoam', cwd=CASE_DIR)

In [ ]:
# ── Step 10: Extract and plot forces ──────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import re

force_glob = os.path.join(CASE_DIR, 'postProcessing', 'propellerForces')

def read_forces_notebook(case_dir):
    pb = os.path.join(case_dir, 'postProcessing', 'propellerForces')
    if not os.path.isdir(pb):
        return None
    time_dirs = sorted([d for d in os.listdir(pb) if os.path.isdir(os.path.join(pb, d))],
                        key=lambda x: float(x) if x.replace('.','').isdigit() else 0)
    if not time_dirs:
        return None
    ff = os.path.join(pb, time_dirs[-1], 'force.dat')
    if not os.path.isfile(ff):
        return None
    data = []
    with open(ff) as f:
        for line in f:
            if line.startswith('#') or not line.strip(): continue
            nums = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', line)
            if len(nums) >= 13:
                data.append([float(x) for x in nums[:13]])
    if not data: return None
    arr = np.array(data)
    return arr[:, 0], arr[:, 3] + arr[:, 6], arr[:, 11] + arr[:, 12]

result = read_forces_notebook(CASE_DIR)

if result is not None:
    t_iters, Fz, Mz = result
    T_converged = Fz[-50:].mean() / 9.81 * 1000
    Q_converged = Mz[-50:].mean()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(t_iters, Fz / 9.81 * 1000)
    ax1.axhline(T_converged, color='red', ls='--', label=f'Converged: {T_converged:.1f} gf')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('Thrust [gf]')
    ax1.set_title('Thrust Convergence'); ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(t_iters, Mz * 1000)
    ax2.axhline(Q_converged * 1000, color='red', ls='--', label=f'Converged: {Q_converged*1000:.2f} mN·m')
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Torque [mN·m]')
    ax2.set_title('Torque Convergence'); ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/plots/cfd_convergence.png', dpi=150)
    plt.show()

    print(f'\n=== CFD Result ({CONFIG}, {RPM} RPM) ===')
    print(f'  Thrust : {T_converged:.1f} gf = {T_converged*9.81/1000:.3f} N')
    print(f'  Torque : {Q_converged*1000:.2f} mN·m')

    # Compare with BEMT
    from bemt.bemt_solver import BEMTSolver
    from bemt.propeller_configs import CONFIGS
    solver = BEMTSolver(CONFIGS[CONFIG], n_sections=60)
    bemt_r = solver.solve(RPM, V_INF)
    print(f'  BEMT T : {bemt_r["T"]/9.81*1000:.1f} gf')
    print(f'  Diff   : {(T_converged - bemt_r["T"]/9.81*1000)/bemt_r["T"]*9.81*1000*100:+.1f}%  (CFD vs BEMT)')
else:
    print('Force data not found — simulation may not have completed yet.')
    print('Run Allrun script manually or check log files.')

## Next Steps

### Run all cases
```bash
# Generate all 48 cases (4 configs × 3 RPMs × 4 wind speeds)
bash scripts/setup_all_cases.sh

# Run all cases (sequential on local machine)
bash scripts/run_all_cases.sh

# Post-process all cases
python postprocessing/extract_forces.py --runs_dir runs/

# Generate comparison plots (CFD + BEMT + Experimental)
python postprocessing/plot_comparison.py --source both --runs_dir runs/
```

### Replace parametric geometry with actual CAD
1. Export your propeller CAD as STL (units: metres)
2. Copy to `geometry/stl/<config_name>.stl`
3. Re-run the case setup and simulation

### Increase mesh resolution
Edit `openfoam/templates/*/system/snappyHexMeshDict`:
- Increase `level` values in `refinementSurfaces` (e.g., `(6 8)` for finer near-wall mesh)
- Increase background grid in `blockMeshDict` (e.g., `(30 30 40)` for denser background)

### Transient simulation (sliding mesh)
Replace `simpleFoam` with `pimpleFoam` and change MRF to AMI (Arbitrary Mesh Interface) for higher-fidelity time-resolved predictions.